# Analysis of Dispatch vs. Demand
## 1. Introduction

This Notebook investigates how dispatch from models satisfies demand. We'll load various datasets, perform comparisons, and analyze results through visualizations.

## 2. Import Libraries

In [43]:
# Import necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import swcol as sw
parse_tech, colors, tech_order, tech_colors = sw.template.get() # Template

## 3. Load and Prepare Data

In [44]:
model_inputs_path = '../../model/inputs/'
model_outputs_path = '../../model/outputs/'
dema_path = '../../data/XM-API/variable_query/2022-12-01_2023-11-30/'
base_year = '2023'

### 3.1 Timepoints
It is always advisable to load the timepoints from the model to simplify the transformations of many tables when merging.

In [45]:
# Load timepoints
timepoints = pd.read_csv(model_inputs_path+'timepoints.csv')
timepoints['timepoints'] = timepoints['timepoint_id']
timepoints = timepoints.drop(columns=['timeseries','timepoint_id'])

### 3.2.1 Dispatch by Tecnology from Switch
`dispatch_system` will be compared with the system registry on xm, while `dispatch` will be used to compare by technology.

In [46]:
dispatch_sw = pd.read_csv(model_outputs_path+'dispatch.csv')
dispatch_sw = dispatch_sw.groupby(['timestamp','gen_tech']).agg({
    'DispatchGen_MW' : 'sum'
}).reset_index()

dispatch_sw = pd.merge(dispatch_sw, timepoints, on='timestamp', how='inner')
dispatch_sw = dispatch_sw.sort_values(by='timepoints')

dispatch_sw.head(3)

,timestamp,gen_tech,DispatchGen_MW,timepoints
120,2023_Q1_labor_0h,Eolica,39.970764,1
121,2023_Q1_labor_0h,Hidro,6804.846265,1
124,2023_Q1_labor_0h,pv_solar,0.000000,1


### 3.2.1 Dispatch by Tecnology from XM

In [47]:
dispatch_xm = pd.read_csv(dema_path+'Melted_Gen_Res.csv')
dispatch_xm['DispatchGen_MW'] = dispatch_xm['GeneReal']
# Add Timepoints
dispatch_xm = pd.merge(dispatch_xm, timepoints, on='timestamp', how='inner')
# Sort by timepoints
dispatch_xm = dispatch_xm.sort_values(by='timepoints')

dispatch_xm.head(3)

,timestamp,Tech,GeneReal,DispatchGen_MW,timepoints
114,2023_Q1_labor_0h,Wind,21.928230,21.928230,1
113,2023_Q1_labor_0h,Thermal,1605.389351,1605.389351,1
112,2023_Q1_labor_0h,Solar,0.000645,0.000645,1


### 3.3.1 Dispatch by System from Switch

In [48]:
dispatch_sys = dispatch_sw.groupby(['timestamp','timepoints']).agg({
    'DispatchGen_MW' : 'sum'
    }).reset_index()

### 3.3.2 Demand by System from Switch

In [49]:
loads = pd.read_csv(model_inputs_path+'loads.csv')
loads = loads.groupby(['timepoints']).agg({
    'zone_demand_mw': 'sum'
}).reset_index()

print(loads.shape)
loads.head(3)

(1536, 2)


,timepoints,zone_demand_mw
0,1,8433.925584
1,2,7961.776883
2,3,7689.767922


### 3.3.3 Dispatch and Demand by System from XM

In [50]:
XM = pd.read_csv(dema_path+'Melted_DemaGen_Sys.csv')
# Group by timestamp
XM = XM.groupby(['timestamp']).agg({
    'DemaReal_Sistema': 'mean', 'DemaCome_Sistema': 'mean',
    'Gene_Sistema': 'mean', 'GeneIdea_Sistema': 'mean'
}).reset_index()
# Adjust to MW
XM['DemaReal_Sistema'] = XM['DemaReal_Sistema'] / 1000
XM['DemaCome_Sistema'] = XM['DemaCome_Sistema'] / 1000
XM['Gene_Sistema'] = XM['Gene_Sistema'] / 1000
XM['GeneIdea_Sistema'] = XM['GeneIdea_Sistema'] / 1000

print(XM.shape)
XM.head(3)

(192, 5)


,timestamp,DemaReal_Sistema,DemaCome_Sistema,Gene_Sistema,GeneIdea_Sistema
0,2023_Q1_holidays_0h,7915.745849,8076.836295,8076.777236,8076.777236
1,2023_Q1_holidays_10h,7725.791177,7856.700819,7856.700819,7856.700819
2,2023_Q1_holidays_11h,8022.721860,8154.997578,8154.919696,8154.919696


## 3.4 System Analysis

### 3.4.1 Is Demand on Switch met by Switch Dispatch?

In [51]:
# Merge tables, result will be: timepoints, timestamp, zone_demand_mw, dispatch_wide 
demand_x_dispatch = pd.merge(loads, dispatch_sys, on='timepoints', how='inner')
# Sort by Timepoints
demand_x_dispatch = demand_x_dispatch.sort_values(by='timepoints')
# Rename columns for better undestanding
demand_x_dispatch.rename(columns={'DispatchGen_MW': 'Dispatch', 'zone_demand_mw': 'Demand'}, inplace=True)

sw.dispatch.plot_dispatch_vs_demand(demand_x_dispatch, 'Demand/Dispatch XM vs Dispatch Switch', ['Dispatch', 'Demand'])

### 3.4.2 Are Demand and Dispatch on XM similar to Switch Dispatch?

In [52]:
# Merge tables
demand_x_dispatch = pd.merge(XM, dispatch_sys, on='timestamp', how='inner')
import plotly.express as px
# Sort by timepoints to sort timestamps as well
demand_x_dispatch = demand_x_dispatch.sort_values(by='timepoints')

sw.dispatch.plot_dispatch_vs_demand(demand_x_dispatch, 'Demand/Dispatch XM vs Dispatch Switch', ['DemaReal_Sistema', 'DemaCome_Sistema', 'Gene_Sistema', 'GeneIdea_Sistema', 'DispatchGen_MW'])

### 3.4.3 Dispatch by Technology XM

In [53]:
gen_res = dispatch_xm.copy()
# Replace prefix 'labor_' by 'L_' and 'holidays_' by 'H_'
gen_res['timestamp'] = gen_res['timestamp'].str.replace('labor_', 'L_')
gen_res['timestamp'] = gen_res['timestamp'].str.replace('holidays_', 'H_')

sw.dispatch.plot_line(gen_res, 'Generation by Technology (XM)', {'DispatchGen_MW': 'Reported Generation [MW]'})

### 3.4.4 Dispatch by Technology Switch

In [54]:
gen_disp = dispatch_sw.copy()
# Add Timepoints
gen_disp = pd.merge(gen_disp, timepoints, on='timestamp', how='inner')
# Sort by timepoints to sort timestamps as well
gen_disp = gen_disp.sort_values(by='timepoints_x')
gen_disp.rename(columns={'timepoints_x': 'timepoints'}, inplace=True)

# Change values on gen_tech to match Switch output
gen_disp['gen_tech'] = gen_disp['gen_tech'].replace(parse_tech)
# Replace prefix 'labor_' by 'L_' and 'holidays_' by 'H_'
gen_disp['timestamp'] = gen_disp['timestamp'].str.replace('labor_', 'L_')
gen_disp['timestamp'] = gen_disp['timestamp'].str.replace('holidays_', 'H_')

gen_disp['Tech'] = gen_disp['gen_tech']

sw.dispatch.plot_line(gen_disp, 'Generation by Technology (Switch)', {'DispatchGen_MW': 'Reported Generation [MW]'})

In [55]:
dispatch_tx = pd.read_csv(model_outputs_path+'DispatchTx.csv')

sw.dispatch.plot_sankey(dispatch_tx, model_inputs_path, model_outputs_path)

In [56]:
sw.dispatch.plot_sankey_timestamp(dispatch_tx, timepoints, model_inputs_path, model_outputs_path)

Dropdown(description='Year-Month:', options=('2023-01', '2023-04', '2023-07', '2023-10', '2025-01', '2025-04',…

Dropdown(description='Hour:', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2…

Button(description='Update Sankey', style=ButtonStyle())

### 3.4.5 Dispatch by Technology Switch x XM

In [57]:
gen_res = gen_res[['timestamp', 'Tech', 'DispatchGen_MW', ]]
gen_disp = gen_disp[gen_disp['timestamp'].str.startswith(base_year)][['timestamp', 'Tech', 'DispatchGen_MW']]

sw.dispatch.plot_base_year_comparission(gen_res, gen_disp, base_year='2023')

In [58]:
sw.dispatch.plot_q1_generation(gen_res, gen_disp)

In [59]:
sw.dispatch.plot_generation_per_typical_day(gen_res, gen_disp)

In [60]:
# Align dataframes based on 'timestamp', 'Tech'
gen_res['DispatchGen_MW_xm'] = gen_res['DispatchGen_MW']
gen_disp['DispatchGen_MW_sw'] = gen_disp['DispatchGen_MW']
gen_compare = pd.merge(gen_res, gen_disp, on=['timestamp', 'Tech'])
# Calculate error percentage
gen_compare['percentage_error'] = 100*((gen_compare['DispatchGen_MW_sw'] / gen_compare['DispatchGen_MW_xm']) - 1)
# Remove rows where 'type' contains '_0h' and 'tech' is 'solar'
hours = r'_19h|_20h|_21h|_22h|_23h|_0h|_1h|_2h|_3h|_4h|_5h|_6h|_7h'
condition = ((gen_compare['Tech'] == 'Solar') & (gen_compare['timestamp'].str.contains(hours)))
gen_compare.loc[condition, ['percentage_error']] = np.nan
#print(gen_compare.head(5))

import plotly.express as px
fig = px.line(
    gen_compare,
    x='timestamp', y='percentage_error', 
    color='Tech', color_discrete_map=tech_colors, category_orders={"Tech": tech_order},
    labels={"percentage_error": "Percentage Error [%]", "timestamp":"Timestamp"},
    height=9*50, width=16*50, template="plotly_white")
fig.update_layout(yaxis=dict(range=[-100, 150]))
fig.update_xaxes(dtick=6)
fig.write_image("../images/Percentage Error.png")
fig.show()

### Error by Quartil

In [61]:
# Align dataframes based on 'timestamp', 'Tech'
gen_compare_Q = pd.merge(gen_res, gen_disp, on=['timestamp', 'Tech'])
gen_compare_Q['Quarter'] = gen_compare_Q['timestamp'].apply(lambda x: x[:7])

gen_compare_Q = gen_compare_Q.groupby(['Quarter','Tech']).agg({
    'DispatchGen_MW_xm': 'mean', 'DispatchGen_MW_sw' : 'mean'
}).reset_index()

# Calculate error percentage
gen_compare_Q['percentage_error'] = 100 * ((gen_compare_Q['DispatchGen_MW_sw'] / gen_compare_Q['DispatchGen_MW_xm']) -1)
#print(gen_compare_Q.head(5))
import plotly.express as px
fig = px.line(
    gen_compare_Q, 
    x='Quarter', y='percentage_error', 
    color='Tech', color_discrete_map=tech_colors, category_orders={"Tech": tech_order},
    labels={"percentage_error": "Percentage Error [%]"},
    height=9*50, width=16*50, template="plotly_white")
fig.write_image("../images/Percentage Error Anual.png")
fig.show()

In [62]:
# Alinear los dataframes usando merge
gen_compare_year = pd.merge(gen_res, gen_disp, on=['timestamp', 'Tech'])

gen_compare_year = gen_compare_Q.groupby(['Tech']).agg({
    'DispatchGen_MW_xm': 'mean', 'DispatchGen_MW_sw' : 'mean'
}).reset_index()

# Calcular el porcentaje
gen_compare_year['percentage'] = (gen_compare_year['DispatchGen_MW_sw'] / gen_compare_year['DispatchGen_MW_xm']) - 1
print('Error by Tech (Year)')
gen_compare_year

Error by Tech (Year)


,Tech,DispatchGen_MW_xm,DispatchGen_MW_sw,percentage
0,Hydro,6229.754328,6583.731273,0.056820
1,Run of River,383.952418,522.333663,0.360412
2,Solar,146.755575,201.397949,0.372336
3,Thermal,1925.710945,4062.697457,1.109713
4,Wind,19.660513,36.375772,0.850194


In [63]:
# Promedio ponderado de GeneReal y percentage
weighted_avg_percentage = \
    (gen_compare_year['DispatchGen_MW_xm'] * gen_compare_year['percentage']).sum() / gen_compare_year['DispatchGen_MW_xm'].sum()
print(f"Average error percentage: {weighted_avg_percentage*100:.6f}%")

Average error percentage: 31.021754%
